
# Evaluate Upfront Cost Predictions on ResStock 2024.2 Data

### Goal
We want to understand the predictions that the current version of the model is outputting before putting that model into production. So far, this script is quite basic and shows only a few plots that look at the monotonocity of predicted cost in relation to square footage and heat pump tonnage. However, other such plots should be added here going forward to montior model performance.

### I/O
##### Inputs:
- `upfront_cost.hvac_training_features_{CURRENT_POETRY_VERISON}`
- `upfront_cost_prediction.ashp_upfront_cost_by_building_{RELEASE_VERSION}`

### Import Packages

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyspark.sql import functions as F
import seaborn as sns

from src import util

### Load Constants

In [0]:
dbutils.widgets.dropdown(
    "mode", "feature", ["feature", "develop", "main", "hotfix"]
)  # mode to read/write delta tables in (see `util.modify_db_table_name()`)
dbutils.widgets.text(
    "release_version", ""
)  # release version number to tag outputs tables with: ignored unless Mode=Main, and passed in by job only
MODE = dbutils.widgets.get("mode")
print(MODE)

In [0]:
CURRENT_POETRY_VERISON = util.get_current_semantic_version_from_poetry()

### Load Data

In [0]:
training_features = spark.table(
    util.modify_db_table_name(f"upfront_cost.hvac_training_features_{CURRENT_POETRY_VERISON}", mode=MODE)
)

training_features = training_features.filter(F.col("project_cost_adjusted") < 100000).withColumn(
    "sqft_bin",
    F.when(F.col("sqft") < 1500, "0-1499")
    .when(F.col("sqft") < 2500, "1500-2499")
    .when(F.col("sqft") < 5500, "2500-5499")
    .otherwise("5500+"),
)

training_features_pd = training_features.toPandas()

In [0]:
resstock_predictions = spark.table(
    util.modify_db_table_name(f"upfront_cost_prediction.ashp_upfront_cost_by_building_{CURRENT_POETRY_VERISON}", MODE)
)

resstock_predictions_pd = resstock_predictions.toPandas()

### Analyze Monotonicity of Costs on Square Footage and Heat Pump Tonnage

#### In Training Data

In [0]:
# Create a crosstab
ct = pd.crosstab(
    index=training_features_pd["sqft_bin"],
    columns=training_features_pd["heating_capacity_tons"],
    values=training_features_pd["project_cost_adjusted"],
    aggfunc="mean",
)

# Plot the crosstab as a heatmap
ax = sns.heatmap(ct, cmap="YlGnBu", annot=False)

# Number of columns in the heatmap
n_cols = ct.shape[1]

# Create 8 evenly spaced tick positions across the heatmap width (corresponding to hp tonnage values)
tick_positions = np.linspace(0, n_cols, 8)
tick_labels = [str(i) for i in range(8)]

# Apply them to the bottom axis
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels)

plt.title("Costs across Home and Heat Pump Size (Training Data)")
plt.ylabel("Square Footage")
plt.xlabel("Heat Pump Size (Tons)")
plt.show()

#### In ResStock 2024 Predictions

In [0]:
# Create a crosstab
ct = pd.crosstab(
    index=resstock_predictions_pd["in_sqft_bin"],
    columns=resstock_predictions_pd["heating_capacity_tons"],
    values=resstock_predictions_pd["expected_upfront_cost"],
    aggfunc="mean",
)

# Plot the crosstab as a heatmap
ax = sns.heatmap(ct, cmap="YlGnBu", annot=False)

# Number of columns in the heatmap
n_cols = ct.shape[1]

# Create 15 evenly spaced tick positions across the heatmap width
tick_positions = np.linspace(0, n_cols, 15)
tick_labels = [str(i) for i in range(15)]

# Apply them to the bottom axis
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels)

plt.title("Expected Costs across Home and Heat Pump Size (ResStock 2024)")
plt.ylabel("Square Footage")
plt.xlabel("Heat Pump Size (Tons)")
plt.show()